In [2]:
import numpy as np
import pandas as pd

class LimitOrderBookGenerator:
    def __init__(self, tick_size=0.01, n_levels=20, min_qty=1, max_qty=10):
        """
        Initialize the limit order book generator.

        :param tick_size: Minimum price increment.
        :param n_levels: Number of bid and ask price levels.
        :param min_qty: Minimum order quantity at each level.
        :param max_qty: Maximum order quantity at each level.
        """
        self.tick_size = tick_size
        self.n_levels = n_levels
        self.min_qty = min_qty
        self.max_qty = max_qty
        self.current_timestamp = 0
        self.order_book = {
            "bids": {i * tick_size: np.random.randint(min_qty, max_qty + 1) for i in range(100, 80, -1)},
            "asks": {i * tick_size: np.random.randint(min_qty, max_qty + 1) for i in range(101, 121)}
        }
        self.history = []  # Store historical states of the order book
        self.params_market = None
        self.params_limit = None
        self.params_cancel = None

    def generate_lob(self):
        """
        Generate the current state of the limit order book with the timestamp.
        """
        bids = {f"bid_price{i+1}": price for i, price in enumerate(self.order_book["bids"].keys())}
        bid_qtys = {f"bid_quantity{i+1}": qty for i, qty in enumerate(self.order_book["bids"].values())}

        asks = {f"ask_price{i+1}": price for i, price in enumerate(self.order_book["asks"].keys())}
        ask_qtys = {f"ask_quantity{i+1}": qty for i, qty in enumerate(self.order_book["asks"].values())}

        lob = {"timestamp": self.current_timestamp, **bids, **bid_qtys, **asks, **ask_qtys}
        return lob

    def process_market_orders(self, spread, volume_at_best):
        """
        Handle market orders.
        """
        lambda_market = self.intensity_market(spread, volume_at_best)
        num_market_orders = int(np.round(lambda_market))

        for _ in range(num_market_orders):
            order_size = self.generate_marketorder_size(3)  # Example median size for orders
            if np.random.rand() < 0.5:  # Simulate buy market order
                best_ask = min(self.order_book["asks"].keys())
                while order_size > 0 and best_ask in self.order_book["asks"]:
                    if self.order_book["asks"][best_ask] >= order_size:
                        self.order_book["asks"][best_ask] -= order_size
                        if self.order_book["asks"][best_ask] == 0:
                            del self.order_book["asks"][best_ask]
                        order_size = 0
                    else:
                        order_size -= self.order_book["asks"][best_ask]
                        del self.order_book["asks"][best_ask]
            else:  # Simulate sell market order
                best_bid = max(self.order_book["bids"].keys())
                while order_size > 0 and best_bid in self.order_book["bids"]:
                    if self.order_book["bids"][best_bid] >= order_size:
                        self.order_book["bids"][best_bid] -= order_size
                        if self.order_book["bids"][best_bid] == 0:
                            del self.order_book["bids"][best_bid]
                        order_size = 0
                    else:
                        order_size -= self.order_book["bids"][best_bid]
                        del self.order_book["bids"][best_bid]

    def process_limit_orders(self, spread, total_volume):
        """
        Handle limit orders.
        """
        lambda_limit = self.intensity_limit(spread, total_volume)
        num_limit_orders = int(np.round(lambda_limit))

        for _ in range(num_limit_orders):
            order_size = self.generate_marketorder_size(3)  # Example median size for limit orders
            if np.random.rand() < 0.5:  # Simulate buy limit order
                price_offset = abs(self.placement_limit())
                price_level = max(self.order_book["bids"].keys(), default=0) + price_offset
                price_level = round(price_level / self.tick_size) * self.tick_size  # Align to tick size
                if price_level <= min(self.order_book["asks"].keys(), default=float('inf')):  # Ensure no crossing
                    if price_level not in self.order_book["bids"]:
                        self.order_book["bids"][price_level] = 0
                    self.order_book["bids"][price_level] += order_size
            else:  # Simulate sell limit order
                price_offset = abs(self.placement_limit())
                price_level = min(self.order_book["asks"].keys(), default=float('inf')) - price_offset
                price_level = round(price_level / self.tick_size) * self.tick_size  # Align to tick size
                if price_level >= max(self.order_book["bids"].keys(), default=0):  # Ensure no crossing
                    if price_level not in self.order_book["asks"]:
                        self.order_book["asks"][price_level] = 0
                    self.order_book["asks"][price_level] += order_size

    def process_cancellations(self):
        """
        Handle cancellations of orders.
        """
        for _ in range(np.random.poisson(1)):
            side = "asks" if np.random.rand() < 0.5 else "bids"
            if len(self.order_book[side]) > 0:
                price_level = np.random.choice(list(self.order_book[side].keys()))
                self.order_book[side][price_level] = max(0, self.order_book[side][price_level] - np.random.poisson(3))
                if self.order_book[side][price_level] == 0:
                    del self.order_book[side][price_level]

    def update_order_book(self):
        """
        Update the order book based on market orders, limit orders, and cancellations.
        """
        if not self.order_book["asks"] or not self.order_book["bids"]:  # Reinitialize if either side is empty
            self.order_book["bids"] = {i * self.tick_size: np.random.randint(1, 10) for i in range(100, 80, -1)}
            self.order_book["asks"] = {i * self.tick_size: np.random.randint(1, 10) for i in range(101, 121)}
            return

        spread = min(self.order_book["asks"].keys()) - max(self.order_book["bids"].keys())
        volume_at_best = self.order_book["asks"][min(self.order_book["asks"].keys())]
        total_volume = sum(self.order_book["asks"].values())

        self.process_market_orders(spread, volume_at_best)
        self.process_limit_orders(spread, total_volume)
        self.process_cancellations()

        # Record the updated state of the order book
        self.history.append(self.generate_lob())
        self.current_timestamp += 1

    def intensity_market(self, spread, volume_at_best):
        """
        Calculate the market order intensity based on the model in the paper.
        """
        beta = self.params_market
        return np.exp(
            beta["β0"] + beta["β1"] * np.log(spread + 1e-5) + beta["β11"] * np.log(spread + 1e-5)**2 +
            beta["β2"] * np.log(1 + volume_at_best) + beta["β22"] * np.log(1 + volume_at_best)**2 +
            beta["β12"] * np.log(spread + 1e-5) * np.log(1 + volume_at_best)
        )

    def generate_marketorder_size(self, sigma_median):
        """
        Generate a market order size based on a given median value.
        """
        return int(np.random.exponential(sigma_median))

    def intensity_limit(self, spread, total_volume):
        """
        Calculate the limit order intensity based on the model in the paper.
        """
        β = self.params_limit
        return np.exp(
            β["β0"] + β["β1"] * np.log(spread + 1e-5) + β["β11"] * np.log(spread + 1e-5)**2 +
            β["β2"] * np.log(1 + total_volume) + β["β22"] * np.log(1 + total_volume)**2 +
            β["β12"] * np.log(spread + 1e-5) * np.log(1 + total_volume)
        )

    def placement_limit(self):
        """
        Calculate the price distribution for limit orders based on the model in the paper.
        """
        components = self.params_limit["placement"]
        probabilities = components["weights"]
        means = components["means"]
        stds = components["stds"]

        # Sample prices based on a Gaussian Mixture Model
        component = np.random.choice(len(probabilities), p=probabilities)
        price_offset = np.random.normal(means[component], stds[component])
        return price_offset

    def view_lob_history(self):
        """
        View the entire limit order book history.
        """
        df = pd.DataFrame(self.history)
        print(df)

    def save_to_csv(self, filename="limit_order_book.csv"):
        """
        Save the history of the limit order book to a CSV file.
        """
        df = pd.DataFrame(self.history)
        df.to_csv(filename, index=False)
        print(f"Limit order book history saved to '{filename}'")

# Example usage
params_market = {
    "β0": 0.1, "β1": -0.5, "β11": 0.1, "β2": -0.3, "β22": 0.05, "β12": 0.01
}

params_limit = {
    "β0": 0.2, "β1": 0.4, "β11": -0.1, "β2": -0.2, "β22": 0.1, "β12": -0.05,
    "placement": {
        "weights": [0.3, 0.4, 0.3],
        "means": [0, 2, 5],
        "stds": [0.1, 0.5, 1.0]
    }
}

generator = LimitOrderBookGenerator(tick_size=0.01, n_levels=20, min_qty=1, max_qty=10)
generator.params_market = params_market
generator.params_limit = params_limit

for _ in range(5):  # Simulate 5 updates
    generator.update_order_book()

generator.view_lob_history()
generator.save_to_csv()


   timestamp  bid_price1  bid_price2  bid_price3  bid_price4  bid_price5  \
0          0        0.90        0.89        0.88        0.87        0.86   
1          1        0.89        0.88        0.87        0.86        0.85   
2          2        0.89        0.88        0.87        0.86        0.85   
3          3        0.89        0.88        0.87        0.86        0.85   
4          4        0.89        0.88        0.87        0.86        0.85   

   bid_price6  bid_price7  bid_price8  bid_price9  ...  ask_price9  \
0        0.85        0.84        0.83        0.82  ...         1.2   
1        0.84        0.83        0.82        0.81  ...         1.2   
2        0.84        0.83        0.82        0.81  ...         NaN   
3        0.84        0.83        0.82        0.81  ...         NaN   
4        0.84        0.83        0.82        0.81  ...         NaN   

   ask_quantity1  ask_quantity2  ask_quantity3  ask_quantity4  ask_quantity5  \
0              9              4           

In [ ]:
def generate_marketorder_size(sigma_median):
    """
    Generates a market order size based on an exponential distribution.

    """
    size = np.random.exponential(scale=sigma_median)
    return max(1, int(round(size)))

# Example usage
sigma_median = 50 # Should be evaluated from historical data
order_size = generate_marketorder_size(sigma_median)
print(f"Generated market order size: {order_size}")

Generated market order size: 15
